In [ ]:
import os, json
import pandas as pd
from features import COMBOS, load_reference, reference_scales
from loro_eval import run_loro, evaluate, by_run_table, run_shuffle, summarize_shuffle
from scope import apply_scope, scope_tag

# ---------------- 범위 설정
EXCLUDE_INITIAL = True       # 초기허용불량 제거
EXCLUDE_TRANSIENT = False    # 재가동 후 5샷 전체 제거 (원하면 True로 한 번 더 돌려 비교)
TAG = scope_tag(EXCLUDE_INITIAL, EXCLUDE_TRANSIENT)
scope = lambda d: apply_scope(d, EXCLUDE_INITIAL, EXCLUDE_TRANSIENT)

OUT = f'./outputs/loro_{TAG}'
os.makedirs(OUT, exist_ok=True)
UNLABELED = '../../1. 사출성형기 AI 데이터셋/dataset/unlabeled_data.csv'   # 경로는 각자 맞게
COMBO_LIST = list(COMBOS)
MODELS = ['rule', 'logit', 'tree', 'iforest']

# ---------------- 기준값 캐시
SCALE_PATH = './base/reference_scales.json'
if os.path.exists(SCALE_PATH):
    SCALES = json.load(open(SCALE_PATH, encoding='utf-8'))
else:
    SCALES = {}
    for p in ['cn7', 'rg3']:
        steps, fr = reference_scales(load_reference(UNLABELED, p))
        SCALES[p] = {'steps': steps, 'fr_step': float(fr)}
    json.dump(SCALES, open(SCALE_PATH, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

# ---------------- 데이터 (파생변수는 전체 순서로 계산된 것을 읽고, 범위 필터만 적용)
FEAT, SHOTS = {}, {}
for p in ['cn7', 'rg3']:
    FEAT[p] = scope(pd.read_csv(f'./base/{p}_features.csv', parse_dates=['TimeStamp']))
    sh = pd.read_csv(f'./base/{p}_shots.csv', parse_dates=['TimeStamp'])
    sh = sh[sh['split'] != '제외'].copy(); sh['Reason'] = sh['Reason'].fillna('정상')
    SHOTS[p] = sh                                         # 셔플용 원본 (필터는 피처 계산 후)
    d = FEAT[p]
    print(f"{p.upper()} [{TAG}] 샷 {len(d)}, 불량 {int(d['PassOrFail'].sum())}",
          d.loc[d['PassOrFail'] == 1, 'Reason'].value_counts().to_dict(),
          '| 불량 있는 Run:', sorted(d.loc[d['PassOrFail'] == 1, 'run_id'].unique()))

# ---------------- 1) LORO
OOF = []
for p in ['cn7', 'rg3']:
    print(p, '...')
    OOF.append(run_loro(FEAT[p], p, COMBO_LIST, MODELS, label_mode='all'))
OOF = pd.concat(OOF, ignore_index=True)

# ---------------- 2) 셔플 대조 (같은 범위로)
for p in ['cn7', 'rg3']:
    print(p, 'shuffle ...')
    OOF = pd.concat([OOF, run_shuffle(SHOTS[p], p, SCALES[p]['steps'], SCALES[p]['fr_step'],
                                      scope_fn=scope)], ignore_index=True)
OOF.to_csv(f'{OUT}/oof.csv', index=False, encoding='utf-8-sig')

# ---------------- 3) 평가
SUMMARY, BY_TYPE = evaluate(OOF)
BY_RUN = by_run_table(OOF[OOF['variant'] == 'original'])
SUMMARY.to_csv(f'{OUT}/summary.csv', index=False, encoding='utf-8-sig')
BY_TYPE.to_csv(f'{OUT}/by_type.csv', index=False, encoding='utf-8-sig')
BY_RUN.to_csv(f'{OUT}/by_run.csv', index=False, encoding='utf-8-sig')
print('저장:', os.path.abspath(OUT))

cn7 all ...
cn7 process ...
rg3 all ...
cn7 shuffle ...
rg3 shuffle ...


In [ ]:
# ---------------- 4) 결과 보기
orig = SUMMARY[SUMMARY['variant'] == 'original']
for (p, es), t in orig.groupby(['product', 'eval_set']):
    print(f'\n===== {p.upper()} / {es}  (불량 {t["n_def"].iloc[0]}, 기본 불량률 {t["base_rate"].iloc[0]:.2%}) =====')
    display(t.pivot_table(index='combo', columns='model', values='R@10%').round(2)
              .style.background_gradient(axis=None, cmap='Blues').format('{:.2f}'))
    display(t.pivot_table(index='combo', columns='model', values='AP').round(3)
              .style.background_gradient(axis=None, cmap='Blues').format('{:.3f}'))

# 유형별 상위 10% 탐지
bt = BY_TYPE[BY_TYPE['variant'] == 'original']
for (p, tname), t in bt.groupby(['product', '유형']):
    print(f'\n--- {p.upper()} / {tname} (n={t["n_def"].iloc[0]}) : hit@10% ---')
    display(t.pivot_table(index='combo', columns='model', values='hit@10%', aggfunc='first'))

# 시계열 효과
display(orig[orig['combo'].str.startswith('T')]
        .pivot_table(index=['product', 'eval_set', 'combo'], columns='model', values=['AP', 'R@10%']).round(3))
display(summarize_shuffle(SUMMARY))

In [ ]:
# ---------------- 5) 최적 조합 추천 (Run별 Recall@10% 우선, 동률이면 AP)
#  ※ 트리의 AP는 fold 간 점수 척도 차이로 낮게 나올 수 있어 Recall@k를 우선 기준으로 둠
cand = orig[(orig['eval_set'] == 'all') & (orig['model'] != 'rule')]
top = (cand.sort_values(['product', 'R@10%', 'R@5%', 'AP'], ascending=[True, False, False, False])
           .groupby('product').head(5))
display(top[['product', 'combo', 'model', 'hit@5%', 'hit@10%', 'AP', 'AP_lo', 'AP_hi']].round(3))

BEST = {p: dict(zip(['combo', 'model'], top[top['product'] == p].iloc[0][['combo', 'model']]))
        for p in ['cn7', 'rg3']}
json.dump(BEST, open(f'{OUT}/best_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('추천:', BEST)
# 규칙 기준선과 비교
display(orig[(orig['model'] == 'rule') & (orig['eval_set'] == 'all')][['product', 'hit@5%', 'hit@10%', 'AP']])